## ATR_nhkのダウンロード

In [ ]:
import os
import gdown
import tarfile

# 1. ダウンロード対象のファイルIDと出力先
# 変更されたURLからファイルIDを抽出
file_id = "1MkqXLcyAIdYQ2VpQQjHXZLShKO9Qcwsc"
output_path = "./downloaded_folder/ATR_nhk.tar"

# 2. 出力先フォルダ作成
os.makedirs("./downloaded_folder", exist_ok=True)

# 3. gdownでダウンロード (ファイルIDを直接指定する形式)
# このURL形式がgdownでのダウンロードに最も適しています。
gdown.download(f"https://drive.google.com/uc?id={file_id}", output_path, quiet=False)

# 4. tarファイルを解凍（拡張子が .tar の場合）
# ダウンロードが正常に完了すれば、この処理が実行されます。
with tarfile.open(output_path, "r") as tar:
    tar.extractall(path="./downloaded_folder")

print("ダウンロードと展開が完了しました。")

In [2]:
import os
import tarfile

# 1. 解凍対象のファイルパスと解凍先フォルダ
# ダウンロードが完了していることを前提とします
tar_file_path = "./downloaded_folder/ATR_nhk.tar"
extraction_path = "./downloaded_folder" # 解凍先フォルダ

# 2. 解凍先フォルダが存在しない場合は作成
os.makedirs(extraction_path, exist_ok=True)

# 3. tarファイルを解凍
try:
    with tarfile.open(tar_file_path, "r") as tar:
        print(f"'{tar_file_path}' の解凍を開始します...")
        tar.extractall(path=extraction_path)
        print(f"'{tar_file_path}' の解凍が '{extraction_path}' に完了しました。")
except FileNotFoundError:
    print(f"エラー: '{tar_file_path}' が見つかりません。ダウンロードが完了しているか確認してください。")
except tarfile.ReadError:
    print(f"エラー: '{tar_file_path}' は有効なtarファイルではないか、破損しています。")
except Exception as e:
    print(f"解凍中に予期せぬエラーが発生しました: {e}")

'./downloaded_folder/ATR_nhk.tar' の解凍を開始します...
'./downloaded_folder/ATR_nhk.tar' の解凍が './downloaded_folder' に完了しました。


In [6]:
import os
import numpy as np
from scipy.io import wavfile
from scipy.signal import find_peaks
import glob
from tqdm import tqdm

# --- ユーザー設定箇所 ---
microphone_prefix_map = {
#    "CM": "CM01",       # センターマイク
    "EM_left": "HM04",  # 耳装着型マイク（左）
    "EM_right": "HM03", # 耳装着型マイク（右）
    "TM_lower": "HM02", # 胸部装着型マイク（下）
    "TM_upper": "HM01", # 胸部装着型マイク（上）
}
# --- ユーザー設定箇所ここまで ---

# 基本パスの定義
base_input_path = "./downloaded_folder/ATR_NHK_corpus/ATR503"
base_output_path = "./../dataset/ATR503/combined"

# 処理対象のマイク種別リストを map から生成
microphone_types = list(microphone_prefix_map.keys())

# 出力ディレクトリの作成
os.makedirs(base_output_path, exist_ok=True)

print("音声ファイルの加算処理を開始します。\n")
print(f"処理対象マイク: {', '.join(microphone_types)}")
print(f"入力元: {base_input_path}")
print(f"出力先: {base_output_path}\n")

# 処理対象のマイクが指定されているか確認
if not microphone_types:
    print("エラー: 処理対象のマイクが指定されていません。`microphone_prefix_map` を編集してください。")
    exit()

try:
    # リストの最初のマイクを基準としてファイルのリストアップを行う
    reference_mic_type = microphone_types[0]
    reference_mic_path = os.path.join(base_input_path, reference_mic_type)
    reference_prefix = microphone_prefix_map[reference_mic_type]

    reference_wav_pattern = os.path.join(reference_mic_path, "*.wav")
    all_reference_wavs = glob.glob(reference_wav_pattern)

    if not all_reference_wavs:
        print(f"エラー: 基準パス '{reference_wav_pattern}' 配下にWAVファイルが見つかりません。")
        print(f"基準マイク '{reference_mic_type}' のデータが正しく展開されているか、パスが正しいか確認してください。")
    else:
        print(f"'{len(all_reference_wavs)}' 個の基準WAVファイルを元に処理を開始します。\n")

        # tqdmで処理
        for ref_wav_path in tqdm(all_reference_wavs, desc="加算処理中"):
            ref_filename = os.path.basename(ref_wav_path)

            audio_data_list = []
            sample_rate = None

            # 全てのマイクの音声を読み込む
            for mic_type, prefix in microphone_prefix_map.items():
                current_filename = ref_filename.replace(reference_prefix, prefix)
                current_wav_path = os.path.join(base_input_path, mic_type, current_filename)

                if not os.path.exists(current_wav_path):
                    tqdm.write(f"  情報: ファイル '{current_wav_path}' が見つかりませんでした。スキップします。")
                    continue

                try:
                    sr, data = wavfile.read(current_wav_path)
                    
                    if sample_rate is None:
                        sample_rate = sr
                    elif sr != sample_rate:
                        tqdm.write(f"  警告: サンプリングレートが異なります: {current_wav_path}")
                        continue
                        
                    # ステレオの場合はモノラルに変換
                    if data.ndim > 1:
                        data = np.mean(data, axis=1)
                    
                    audio_data_list.append(data)

                except Exception as e:
                    tqdm.write(f"  エラー: '{current_wav_path}' の読み込みに失敗しました: {e}")
            
            if not audio_data_list:
                tqdm.write(f"  情報: '{ref_filename}' に対応する有効な音声ファイルが見つからなかったため、加算をスキップします。")
                continue

            # 最短の長さに合わせる
            min_length = min(len(data) for data in audio_data_list)
            trimmed_data = [data[:min_length] for data in audio_data_list]

            # 音声データを加算
            combined_data = np.sum(trimmed_data, axis=0)
            
            # クリッピングを防ぐため正規化
            max_val = np.max(np.abs(combined_data))
            if max_val > 0:
                # 元のデータ型の最大値を取得
                if audio_data_list[0].dtype == np.int16:
                    target_max = 32767
                elif audio_data_list[0].dtype == np.int32:
                    target_max = 2147483647
                else:
                    target_max = 1.0
                
                combined_data = combined_data * (target_max * 0.95) / max_val
                combined_data = combined_data.astype(audio_data_list[0].dtype)

            # 出力ファイル名を生成
            output_base_filename = ref_filename.replace(f"_{reference_prefix}", "")
            output_wav_path = os.path.join(base_output_path, output_base_filename)

            try:
                wavfile.write(output_wav_path, sample_rate, combined_data)
            except Exception as e:
                tqdm.write(f"  エラー: 加算された音声 '{output_wav_path}' の保存中にエラーが発生しました: {e}")

except Exception as e:
    print(f"\n全体処理中にエラーが発生しました: {e}")

print("\n音声加算プロセスが完了しました。")

  4%|▍         | 928M/21.1G [20:23<7:23:34, 759kB/s]


音声ファイルの加算処理を開始します。

処理対象マイク: EM_left, EM_right, TM_lower, TM_upper
入力元: ./downloaded_folder/ATR_NHK_corpus/ATR503
出力先: ./../dataset/ATR503/combined

'20120' 個の基準WAVファイルを元に処理を開始します。



加算処理中: 100%|██████████| 20120/20120 [00:09<00:00, 2050.80it/s]


音声加算プロセスが完了しました。


In [7]:
import os
from pydub import AudioSegment
from pydub.exceptions import CouldntDecodeError
import glob
from tqdm import tqdm

# --- ユーザー設定箇所 ---
# 結合（ミキシング）したいマイクの種類をここで定義します。
# 不要なマイクは、行ごとコメントアウトしてください (行頭に # を付けます)。
microphone_identifiers = {
    "CM01": "CM01",  # センターマイク
    "HM01": "HM01",  # TM_upper
    "HM02": "HM02",  # TM_lower
    "HM03": "HM03",  # EM_right
    "HM04": "HM04",  # EM_left
}
# --- ユーザー設定箇所ここまで ---

# 基本パスの定義
base_input_path = "./downloaded_folder/ATR_NHK_corpus/NHK40"
base_output_path = "./../dataset/NHK40/combined"

# 処理対象の識別子リストを map から生成
processing_identifiers = list(microphone_identifiers.keys())

# 出力ディレクトリの作成
os.makedirs(base_output_path, exist_ok=True)

print("NHK40 音声ファイルの結合処理を開始します。\n")
print(f"処理対象マイク: {', '.join(processing_identifiers)}")
print(f"入力元: {base_input_path}")
print(f"出力先: {base_output_path}\n")

if not processing_identifiers:
    print("エラー: 処理対象のマイクが指定されていません。`microphone_identifiers` を編集してください。")
    exit()

try:
    # リストの最初のマイクを基準としてファイルのリストアップを行う
    reference_identifier = processing_identifiers[0]
    reference_path = os.path.join(base_input_path, reference_identifier, "ALL")

    reference_wav_pattern = os.path.join(reference_path, "*.wav")
    all_reference_wavs = glob.glob(reference_wav_pattern)

    if not all_reference_wavs:
        print(f"エラー: 基準パス '{reference_wav_pattern}' 配下にWAVファイルが見つかりません。")
        print(f"基準マイク '{reference_identifier}' のデータが正しく展開されているか、パスが正しいか確認してください。")
    else:
        print(f"'{len(all_reference_wavs)}' 個の基準WAVファイルを元に処理を開始します。\n")

        for ref_wav_path in tqdm(all_reference_wavs, desc="結合処理中"):
            ref_filename = os.path.basename(ref_wav_path)
            
            audio_segments_to_combine = []

            # 全ての指定された識別子の音声を読み込みリストに追加
            for identifier in processing_identifiers:
                current_filename = ref_filename.replace(reference_identifier, identifier)
                current_wav_path = os.path.join(base_input_path, identifier, "ALL", current_filename)

                if not os.path.exists(current_wav_path):
                    tqdm.write(f"  注意: '{current_wav_path}' が見つかりません。この識別子はスキップします。")
                    continue

                try:
                    audio = AudioSegment.from_wav(current_wav_path)
                    audio_segments_to_combine.append(audio)
                except CouldntDecodeError:
                    tqdm.write(f"  エラー: '{current_wav_path}' のデコードに失敗しました。スキップします。")
                except Exception as e:
                    tqdm.write(f"  エラー: '{current_wav_path}' の処理中に予期せぬエラーが発生しました: {e} スキップします。")

            # 結合対象の音声が1つ以上見つかった場合のみ処理
            if len(audio_segments_to_combine) > 0:
                # 最初のオーディオをベースとして設定
                combined_audio = audio_segments_to_combine[0]

                # 2つ目以降のオーディオをオーバーレイ（ミキシング）
                # pydubのoverlayは、サンプルレート等が異なっても自動でベースに合わせてくれます
                if len(audio_segments_to_combine) > 1:
                    for segment_to_overlay in audio_segments_to_combine[1:]:
                        combined_audio = combined_audio.overlay(segment_to_overlay)

                # 出力ファイル名は、基準ファイル名から基準識別子部分を取り除いたもの
                output_base_filename = ref_filename.replace(f"_{reference_identifier}", "")
                output_wav_path = os.path.join(base_output_path, output_base_filename)

                try:
                    combined_audio.export(output_wav_path, format="wav")
                except Exception as e:
                    tqdm.write(f"  結合された音声 '{output_wav_path}' のエクスポート中にエラーが発生しました: {e}")
            else:
                tqdm.write(f"  情報: '{ref_filename}' に対応する有効な音声ファイルが見つからなかったため結合をスキップします。")

except Exception as e:
    print(f"\n全体処理中にエラーが発生しました: {e}")

print("\nNHK40 音声結合プロセスが完了しました。")

NHK40 音声ファイルの結合処理を開始します。

処理対象マイク: CM01, HM01, HM02, HM03, HM04
入力元: ./downloaded_folder/ATR_NHK_corpus/NHK40
出力先: ./../dataset/NHK40/combined

'1600' 個の基準WAVファイルを元に処理を開始します。



結合処理中: 100%|██████████| 1600/1600 [00:01<00:00, 1335.65it/s]


NHK40 音声結合プロセスが完了しました。


In [10]:
import os
import re # 正規表現モジュールをインポート

# 結合済みWAVファイルがあるディレクトリ
combined_audio_path = "./../dataset/ATR503/combined"

print(f"結合済み音声ファイルのファイル名変更を開始します。\n")
print(f"対象ディレクトリ: {combined_audio_path}\n")

# ディレクトリ内の全ての.wavファイルを検索
wav_files = [f for f in os.listdir(combined_audio_path) if f.endswith('.wav')]

if not wav_files:
    print(f"エラー: '{combined_audio_path}' にWAVファイルが見つかりませんでした。パスを確認してください。")
else:
    print(f"'{len(wav_files)}' 個のWAVファイルを処理します。\n")
    processed_count = 0
    skipped_count = 0

    for filename in wav_files:
        # ファイル名から末尾の数字部分を正規表現で抽出
        # 例: ATR503K_div_FMT01_speech001_04.wav
        # 抽出したい部分: 001 と 04
        # 新しいパターン: _speech(\d+)_(\d+)\.wav$
        match = re.search(r'_speech(\d+)_(\d+)\.wav$', filename)

        if match:
            # 抽出した2つの数字グループ
            num1_str = match.group(1) # 例: "001" (speechの後の数字)
            num2_str = match.group(2) # 例: "04" (最後の数字)

            # 変更後の新しいファイル名を生成
            # 例: ATR503K_div_FMT01_speech04_001.wav になるよう、置換する
            # まず、元の '_speechNUM1_NUM2.wav' の部分を特定
            original_suffix_segment = f"speech{num1_str}_{num2_str}.wav"
            # 新しいサフィックスを作成: 'speechNUM2_NUM1.wav'
            new_suffix_segment = f"speech{num2_str}_{num1_str}.wav"

            # ファイル名を置換
            new_filename = filename.replace(original_suffix_segment, new_suffix_segment)

            # 新しいパスと古いパス
            old_filepath = os.path.join(combined_audio_path, filename)
            new_filepath = os.path.join(combined_audio_path, new_filename)

            # ファイル名の変更を実行
            if old_filepath != new_filepath: # 同じファイル名でなければリネーム
                try:
                    os.rename(old_filepath, new_filepath)
                    print(f"変更: '{filename}' -> '{new_filename}'")
                    processed_count += 1
                except OSError as e:
                    print(f"エラー: '{filename}' のリネーム中に問題が発生しました: {e}")
                    skipped_count += 1
            else:
                # 既に正しい形式の場合や、抽出ロジックで変更がない場合
                print(f"スキップ: '{filename}' (変更不要または形式が一致しませんでした)")
                skipped_count += 1
        else:
            print(f"スキップ: '{filename}' (ファイル名のパターンに一致しませんでした)")
            skipped_count += 1

    print(f"\nファイル名変更プロセスが完了しました。")
    print(f"処理済みファイル数: {processed_count}")
    print(f"スキップされたファイル数: {skipped_count}")

結合済み音声ファイルのファイル名変更を開始します。

対象ディレクトリ: ./../dataset/ATR503/combined

'20120' 個のWAVファイルを処理します。

変更: 'ATR503K_div_MKM01_speech007_08.wav' -> 'ATR503K_div_MKM01_speech08_007.wav'
変更: 'ATR503K_div_MSG01_speech047_08.wav' -> 'ATR503K_div_MSG01_speech08_047.wav'
変更: 'ATR503K_div_MYS03_speech029_07.wav' -> 'ATR503K_div_MYS03_speech07_029.wav'
変更: 'ATR503K_div_MKM01_speech002_04.wav' -> 'ATR503K_div_MKM01_speech04_002.wav'
変更: 'ATR503K_div_MSG01_speech047_06.wav' -> 'ATR503K_div_MSG01_speech06_047.wav'
変更: 'ATR503K_div_MRO02_speech001_07.wav' -> 'ATR503K_div_MRO02_speech07_001.wav'
変更: 'ATR503L_div_MKK03_speech038_09.wav' -> 'ATR503L_div_MKK03_speech09_038.wav'
変更: 'ATR503L_div_MYK03_speech000_07.wav' -> 'ATR503L_div_MYK03_speech07_000.wav'
変更: 'ATR503K_div_MRY01_speech041_05.wav' -> 'ATR503K_div_MRY01_speech05_041.wav'
変更: 'ATR503L_div_MTT01_speech039_03.wav' -> 'ATR503L_div_MTT01_speech03_039.wav'
変更: 'ATR503L_div_MHT02_speech034_06.wav' -> 'ATR503L_div_MHT02_speech06_034.wav'
変更: 'ATR503K_div

In [11]:
import os
import re # 正規表現モジュールをインポート

# 結合済みWAVファイルがあるディレクトリ
combined_audio_path = "./../dataset/NHK40/combined"

print(f"結合済み音声ファイルのファイル名変更を開始します。\n")
print(f"対象ディレクトリ: {combined_audio_path}\n")

# ディレクトリ内の全ての.wavファイルを検索
wav_files = [f for f in os.listdir(combined_audio_path) if f.endswith('.wav')]

if not wav_files:
    print(f"エラー: '{combined_audio_path}' にWAVファイルが見つかりませんでした。パスを確認してください。")
else:
    print(f"'{len(wav_files)}' 個のWAVファイルを処理します。\n")
    processed_count = 0
    skipped_count = 0

    for filename in wav_files:
        # ファイル名から末尾の数字部分を正規表現で抽出
        # 例: ATR503K_div_FMT01_speech001_04.wav
        # 抽出したい部分: 001 と 04
        # 新しいパターン: _speech(\d+)_(\d+)\.wav$
        match = re.search(r'_speech(\d+)_(\d+)\.wav$', filename)

        if match:
            # 抽出した2つの数字グループ
            num1_str = match.group(1) # 例: "001" (speechの後の数字)
            num2_str = match.group(2) # 例: "04" (最後の数字)

            # 変更後の新しいファイル名を生成
            # 例: ATR503K_div_FMT01_speech04_001.wav になるよう、置換する
            # まず、元の '_speechNUM1_NUM2.wav' の部分を特定
            original_suffix_segment = f"speech{num1_str}_{num2_str}.wav"
            # 新しいサフィックスを作成: 'speechNUM2_NUM1.wav'
            new_suffix_segment = f"speech{num2_str}_{num1_str}.wav"

            # ファイル名を置換
            new_filename = filename.replace(original_suffix_segment, new_suffix_segment)

            # 新しいパスと古いパス
            old_filepath = os.path.join(combined_audio_path, filename)
            new_filepath = os.path.join(combined_audio_path, new_filename)

            # ファイル名の変更を実行
            if old_filepath != new_filepath: # 同じファイル名でなければリネーム
                try:
                    os.rename(old_filepath, new_filepath)
                    print(f"変更: '{filename}' -> '{new_filename}'")
                    processed_count += 1
                except OSError as e:
                    print(f"エラー: '{filename}' のリネーム中に問題が発生しました: {e}")
                    skipped_count += 1
            else:
                # 既に正しい形式の場合や、抽出ロジックで変更がない場合
                print(f"スキップ: '{filename}' (変更不要または形式が一致しませんでした)")
                skipped_count += 1
        else:
            print(f"スキップ: '{filename}' (ファイル名のパターンに一致しませんでした)")
            skipped_count += 1

    print(f"\nファイル名変更プロセスが完了しました。")
    print(f"処理済みファイル数: {processed_count}")
    print(f"スキップされたファイル数: {skipped_count}")

結合済み音声ファイルのファイル名変更を開始します。

対象ディレクトリ: ./../dataset/NHK40/combined

'1600' 個のWAVファイルを処理します。

変更: 'NHK40K_div_FMT01_speech008_00.wav' -> 'NHK40K_div_FMT01_speech00_008.wav'
変更: 'NHK40L_div_MTT01_speech027_00.wav' -> 'NHK40L_div_MTT01_speech00_027.wav'
変更: 'NHK40K_div_MYK02_speech025_00.wav' -> 'NHK40K_div_MYK02_speech00_025.wav'
変更: 'NHK40L_div_MHT02_speech022_00.wav' -> 'NHK40L_div_MHT02_speech00_022.wav'
変更: 'NHK40K_div_MSM01_speech037_00.wav' -> 'NHK40K_div_MSM01_speech00_037.wav'
変更: 'NHK40L_div_FRT01_speech001_00.wav' -> 'NHK40L_div_FRT01_speech00_001.wav'
変更: 'NHK40K_div_MYN01_speech027_00.wav' -> 'NHK40K_div_MYN01_speech00_027.wav'
変更: 'NHK40K_div_MYK02_speech014_00.wav' -> 'NHK40K_div_MYK02_speech00_014.wav'
変更: 'NHK40K_div_MYN01_speech029_00.wav' -> 'NHK40K_div_MYN01_speech00_029.wav'
変更: 'NHK40L_div_FRT01_speech017_00.wav' -> 'NHK40L_div_FRT01_speech00_017.wav'
変更: 'NHK40K_div_MKM01_speech039_00.wav' -> 'NHK40K_div_MKM01_speech00_039.wav'
変更: 'NHK40K_div_MKJ01_speech025_00.wav' 

### ブランク追加＋json作成


In [12]:
import os
import json
import re
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from pydub.exceptions import CouldntDecodeError
import random
from tqdm import tqdm

# pydubがインストールされていることを前提とします
# pip install pydub
# ffmpegも必要です: apt install ffmpeg (Linux) / brew install ffmpeg (macOS) / Windowsは公式からダウンロードしてください

def create_random_noise_padding(noise_segment, duration_ms):
    """
    【★★★ 機能改善 ★★★】
    指定された長さのノイズパディングを、元ノイズからランダムに断片を抽出して生成します。
    これにより、ノイズの繰り返しパターンを防ぎます。
    """
    if duration_ms <= 0:
        return AudioSegment.empty()
    # 元となるノイズが非常に短いか、存在しない場合は無音を返します。
    if len(noise_segment) < 50:
        return AudioSegment.silent(duration=duration_ms)
    
    padding = AudioSegment.empty()
    # 必要な長さになるまで、元ノイズからランダムな断片を連結します
    while len(padding) < duration_ms:
        # 50msから300msのランダムな長さの断片を切り出します
        max_start = len(noise_segment) - 50
        random_start = random.randint(0, max_start)
        random_length = random.randint(50, 300)
        chunk = noise_segment[random_start : random_start + random_length]
        padding += chunk
        
    # 最終的に正確な長さに切り詰めて返します
    return padding[:duration_ms]


def generate_speech_json_with_blanks(speech_wav_dir, transcript_file_path, output_json_path, processed_wav_output_dir):
    """
    音声ファイルから発話区間を自動検出し(VAD)、その前後に元音声の環境音を付加して、
    高品質な学習用データセットとJSONファイルを作成します。
    """

    # --- 定数 ---
    MAX_FINAL_DURATION_SEC = 15  # 最終的な音声ファイルの最大許容長 (秒)
    # VAD(音声活動検出)のパラメータ
    VAD_SILENCE_THRESH_DB_OFFSET = -16  # 音声のピークから何dB下を無音と見なすか
    VAD_MIN_SILENCE_LEN_MS = 100      # 無音と見なす最小の長さ (ミリ秒)
    #【★★★ 機能改善 ★★★】発話の冒頭が途切れるのを防ぐためのパディング(のりしろ)
    VAD_CHUNK_PADDING_MS = 150        # 検出された発話区間の前後に加えるマージン (ミリ秒)

    # --- ディレクトリの準備 ---
    output_json_dir = os.path.dirname(output_json_path)
    if output_json_dir and not os.path.exists(output_json_dir):
        os.makedirs(output_json_dir)
        print(f"[Info] JSON出力先フォルダを作成しました: {output_json_dir}")
    
    if not os.path.exists(processed_wav_output_dir):
        os.makedirs(processed_wav_output_dir)
        print(f"[Info] 加工済みWAV出力先フォルダを作成しました: {processed_wav_output_dir}")

    # --- 書き起こしファイルの読み込み ---
    transcripts = []
    try:
        with open(transcript_file_path, 'r', encoding='utf-8') as f:
            transcripts = [line.strip() for line in f if line.strip()]
        print(f"[Info] 書き起こしファイルから {len(transcripts)} 行のテキストを読み込みました: {transcript_file_path}")
    except FileNotFoundError:
        print(f"[Error] 書き起こしファイルが見つかりません: {transcript_file_path}")
        return
    except Exception as e:
        print(f"[Error] 書き起こしファイルの読み込み中にエラーが発生しました: {e}")
        return

    # --- 初期化 ---
    output_data = []
    skipped_too_long_count = 0
    skipped_no_transcript_count = 0
    skipped_other_error_count = 0
    skipped_no_speech_detected = 0

    # --- 音声ファイルの処理 ---
    wav_files = sorted([f for f in os.listdir(speech_wav_dir) if f.endswith('.wav')])
    if not wav_files:
        print(f"[Warning] 音声フォルダにWAVファイルが見つかりません: {speech_wav_dir}")
        return

    print(f"[Info] {len(wav_files)} 個のWAVファイルを処理します。")

    for wav_filename in tqdm(wav_files, desc="JSONデータ生成＆音声加工中"):
        # ファイル名からインデックスを抽出
        match_idx = re.search(r'speech(\d+)_(\d+)\.wav$', wav_filename)
        if not match_idx:
            tqdm.write(f"[Warning] ファイル名 '{wav_filename}' のインデックスパターンに一致しませんでした。スキップします。")
            skipped_other_error_count += 1
            continue

        text_yyy, text_xx = map(int, match_idx.groups())
        text_index = text_yyy * 50 + text_xx
        
        if text_index >= len(transcripts):
            tqdm.write(f"[Warning] ファイル '{wav_filename}' に対応するテキスト行 ({text_index + 1}行目) が見つかりません。スキップします。")
            skipped_no_transcript_count += 1
            continue
        transcript = transcripts[text_index]

        # 音声ファイルを読み込み
        wav_path_full = os.path.join(speech_wav_dir, wav_filename)
        try:
            audio = AudioSegment.from_wav(wav_path_full)
        except Exception as e:
            tqdm.write(f"[Error] 音声ファイル '{wav_path_full}' の読込エラー: {e}。スキップします。")
            skipped_other_error_count += 1
            continue
        
        # 音声活動検出(VAD)
        nonsilent_chunks = detect_nonsilent(
            audio,
            min_silence_len=VAD_MIN_SILENCE_LEN_MS,
            silence_thresh=audio.dBFS + VAD_SILENCE_THRESH_DB_OFFSET
        )

        if not nonsilent_chunks:
            tqdm.write(f"[Warning] ファイル '{wav_filename}' で発話区間を検出できませんでした。スキップします。")
            skipped_no_speech_detected += 1
            continue

        #【★★★ 機能改善: VAD精度向上 ★★★】
        # 検出された区間の前後にパディングを追加して、発話の冒頭や末尾が切れるのを防ぐ
        speech_start_ms = max(0, nonsilent_chunks[0][0] - VAD_CHUNK_PADDING_MS)
        speech_end_ms = min(len(audio), nonsilent_chunks[-1][1] + VAD_CHUNK_PADDING_MS)
        speech_audio = audio[speech_start_ms:speech_end_ms]

        if len(speech_audio) / 1000.0 > MAX_FINAL_DURATION_SEC:
            tqdm.write(f"[Warning] ファイル '{wav_filename}' の検出された発話長 ({len(speech_audio)/1000.0:.2f}秒) が制限を超えています。スキップします。")
            skipped_too_long_count += 1
            continue

        # 環境音の抽出
        noise_before = audio[:speech_start_ms]
        noise_after = audio[speech_end_ms:]
        ambient_noise = noise_before + noise_after

        # --- 環境音によるパディング処理 ---
        remaining_capacity_sec = MAX_FINAL_DURATION_SEC - (len(speech_audio) / 1000.0)
        
        # 前方パディング
        front_padding_duration_ms = int(random.uniform(0, remaining_capacity_sec / 2) * 1000)
        front_padding = create_random_noise_padding(ambient_noise, front_padding_duration_ms)
        
        # 後方パディング
        current_len_ms = len(front_padding) + len(speech_audio)
        remaining_for_back_ms = (MAX_FINAL_DURATION_SEC * 1000) - current_len_ms
        back_padding_duration_ms = int(random.uniform(0, remaining_for_back_ms))
        back_padding = create_random_noise_padding(ambient_noise, back_padding_duration_ms)

        processed_audio = front_padding + speech_audio + back_padding
        
        # --- タイムスタンプとパスの計算 ---
        speech_start_sec_final = len(front_padding) / 1000.0
        speech_end_sec_final = speech_start_sec_final + (len(speech_audio) / 1000.0)
        
        # --- 音声ファイルのエクスポート ---
        processed_wav_filepath = os.path.join(processed_wav_output_dir, wav_filename)
        try:
            processed_audio.export(processed_wav_filepath, format="wav")
        except Exception as e:
            tqdm.write(f"[Error] 加工済み音声 '{processed_wav_filepath}' のエクスポート中にエラーが発生しました: {e}。スキップします。")
            skipped_other_error_count += 1
            continue

        # --- JSONデータへの追加 ---
        try:
            abs_wav_path = os.path.abspath(processed_wav_filepath)
            path_parts = abs_wav_path.split(os.sep)
            dataset_index = path_parts.index('dataset')
            base_dir = os.sep.join(path_parts[:dataset_index])
            path_for_json = os.path.relpath(abs_wav_path, start=base_dir)
        except ValueError:
            path_for_json = os.path.relpath(processed_wav_filepath, start=output_json_dir)
        
        path_for_json = path_for_json.replace(os.sep, '/')
        
        item = {
            "path": path_for_json,
            "timestamps": {
                "speech": [[round(speech_start_sec_final, 3), round(speech_end_sec_final, 3)]]
            },
            "text": transcript
        }
        output_data.append(item)

    # --- JSONファイルの書き出し ---
    try:
        with open(output_json_path, 'w', encoding='utf-8') as out_f:
            json.dump(output_data, out_f, ensure_ascii=False, indent=2)
        print(f"\n[Success] JSONファイルを出力しました: {output_json_path}")
    except Exception as e:
        print(f"\n[Error] JSONファイルの書き込み中にエラーが発生しました: {e}")

    # --- 結果のサマリー表示 ---
    print("\n--- 処理結果 ---")
    print(f"合計処理済みファイル数: {len(output_data)}")
    print(f"  - スキップ (長すぎ): {skipped_too_long_count}")
    print(f"  - スキップ (テキストなし): {skipped_no_transcript_count}")
    print(f"  - スキップ (発話検出不可): {skipped_no_speech_detected}")
    print(f"  - スキップ (その他エラー): {skipped_other_error_count}")
    print("--- 処理完了 ---")


# --- 実行例 ---
# ご自身の環境に合わせて、以下のパスを修正してください。
speech_wav_input_dir = "./../dataset/ATR503/combined"
transcript_file_path = "./downloaded_folder/ATR_NHK_corpus/ATR503/ATR503AtoJ.txt"
output_json_path = "./../json/ATR503_speech_labels.json"
processed_wav_output_dir = "./../dataset/ATR503/processed_wavs"

# 関数を呼び出します
print("--- 発話音声用JSON生成と音声加工を開始 ---")
generate_speech_json_with_blanks(speech_wav_input_dir, transcript_file_path, output_json_path, processed_wav_output_dir)


--- 発話音声用JSON生成と音声加工を開始 ---
[Info] 加工済みWAV出力先フォルダを作成しました: ./../dataset/ATR503/processed_wavs
[Info] 書き起こしファイルから 503 行のテキストを読み込みました: ./downloaded_folder/ATR_NHK_corpus/ATR503/ATR503AtoJ.txt
[Info] 20120 個のWAVファイルを処理します。


JSONデータ生成＆音声加工中: 100%|██████████| 20120/20120 [07:26<00:00, 45.10it/s]



[Success] JSONファイルを出力しました: ./../json/ATR503_speech_labels.json

--- 処理結果 ---
合計処理済みファイル数: 20120
  - スキップ (長すぎ): 0
  - スキップ (テキストなし): 0
  - スキップ (発話検出不可): 0
  - スキップ (その他エラー): 0
--- 処理完了 ---


In [1]:
import os
import json
import re
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from pydub.exceptions import CouldntDecodeError
import random
from tqdm import tqdm

# pydubがインストールされていることを前提とします
# pip install pydub
# ffmpegも必要です: apt install ffmpeg (Linux) / brew install ffmpeg (macOS) / Windowsは公式からダウンロードしてください

def create_random_noise_padding(noise_segment, duration_ms):
    """
    指定された長さのノイズパディングを、元ノイズからランダムに断片を抽出して生成します。
    これにより、ノイズの繰り返しパターンを防ぎます。
    """
    if duration_ms <= 0:
        return AudioSegment.empty()
    # 元となるノイズが非常に短いか、存在しない場合は無音を返します。
    if len(noise_segment) < 50:
        return AudioSegment.silent(duration=duration_ms)
    
    padding = AudioSegment.empty()
    # 必要な長さになるまで、元ノイズからランダムな断片を連結します
    while len(padding) < duration_ms:
        # 50msから300msのランダムな長さの断片を切り出します
        max_start = len(noise_segment) - 50
        random_start = random.randint(0, max_start)
        random_length = random.randint(50, 300)
        chunk = noise_segment[random_start : random_start + random_length]
        padding += chunk
        
    # 最終的に正確な長さに切り詰めて返します
    return padding[:duration_ms]


def generate_speech_json_with_blanks(speech_wav_dir, transcript_file_path, output_json_path, processed_wav_output_dir):
    """
    音声ファイルから発話区間を自動検出し(VAD)、その前後に元音声の環境音を付加して、
    高品質な学習用データセットとJSONファイルを作成します。
    """

    # --- 定数 ---
    MAX_FINAL_DURATION_SEC = 15  # 最終的な音声ファイルの最大許容長 (秒)
    # VAD(音声活動検出)のパラメータ
    VAD_SILENCE_THRESH_DB_OFFSET = -16  # 音声のピークから何dB下を無音と見なすか
    VAD_MIN_SILENCE_LEN_MS = 100      # 無音と見なす最小の長さ (ミリ秒)
    VAD_CHUNK_PADDING_MS = 150        # 検出された発話区間の前後に加えるマージン (ミリ秒)

    # --- ディレクトリの準備 ---
    output_json_dir = os.path.dirname(output_json_path)
    if output_json_dir and not os.path.exists(output_json_dir):
        os.makedirs(output_json_dir)
        print(f"[Info] JSON出力先フォルダを作成しました: {output_json_dir}")
    
    if not os.path.exists(processed_wav_output_dir):
        os.makedirs(processed_wav_output_dir)
        print(f"[Info] 加工済みWAV出力先フォルダを作成しました: {processed_wav_output_dir}")

    # --- 書き起こしファイルの読み込み ---
    transcripts = []
    try:
        with open(transcript_file_path, 'r', encoding='utf-8') as f:
            transcripts = [line.strip() for line in f if line.strip()]
        print(f"[Info] 書き起こしファイルから {len(transcripts)} 行のテキストを読み込みました: {transcript_file_path}")
    except FileNotFoundError:
        print(f"[Error] 書き起こしファイルが見つかりません: {transcript_file_path}")
        return
    except Exception as e:
        print(f"[Error] 書き起こしファイルの読み込み中にエラーが発生しました: {e}")
        return

    # --- 初期化 ---
    output_data = []
    skipped_too_long_count = 0
    skipped_no_transcript_count = 0
    skipped_other_error_count = 0
    skipped_no_speech_detected = 0

    # --- 音声ファイルの処理 ---
    wav_files = sorted([f for f in os.listdir(speech_wav_dir) if f.endswith('.wav')])
    if not wav_files:
        print(f"[Warning] 音声フォルダにWAVファイルが見つかりません: {speech_wav_dir}")
        return

    print(f"[Info] {len(wav_files)} 個のWAVファイルを処理します。")

    for wav_filename in tqdm(wav_files, desc="JSONデータ生成＆音声加工中"):
        # ファイル名からインデックスを抽出
        match_idx = re.search(r'speech(\d+)_(\d+)\.wav$', wav_filename)
        if not match_idx:
            tqdm.write(f"[Warning] ファイル名 '{wav_filename}' のインデックスパターンに一致しませんでした。スキップします。")
            skipped_other_error_count += 1
            continue

        text_yyy, text_xx = map(int, match_idx.groups())
        #【注意】データセットによってここの計算が変わる可能性があります
        # ATR503では50個ずつのセットでしたが、NHK40の構成に合わせて調整が必要かもしれません。
        # ここでは一旦同じロジックを流用します。
        text_index = text_yyy * 50 + text_xx
        
        if text_index >= len(transcripts):
            tqdm.write(f"[Warning] ファイル '{wav_filename}' に対応するテキスト行 ({text_index + 1}行目) が見つかりません。スキップします。")
            skipped_no_transcript_count += 1
            continue
        transcript = transcripts[text_index]

        # 音声ファイルを読み込み
        wav_path_full = os.path.join(speech_wav_dir, wav_filename)
        try:
            audio = AudioSegment.from_wav(wav_path_full)
        except Exception as e:
            tqdm.write(f"[Error] 音声ファイル '{wav_path_full}' の読込エラー: {e}。スキップします。")
            skipped_other_error_count += 1
            continue
        
        # 音声活動検出(VAD)
        nonsilent_chunks = detect_nonsilent(
            audio,
            min_silence_len=VAD_MIN_SILENCE_LEN_MS,
            silence_thresh=audio.dBFS + VAD_SILENCE_THRESH_DB_OFFSET
        )

        if not nonsilent_chunks:
            tqdm.write(f"[Warning] ファイル '{wav_filename}' で発話区間を検出できませんでした。スキップします。")
            skipped_no_speech_detected += 1
            continue
        
        # 検出された区間の前後にパディングを追加
        speech_start_ms = max(0, nonsilent_chunks[0][0] - VAD_CHUNK_PADDING_MS)
        speech_end_ms = min(len(audio), nonsilent_chunks[-1][1] + VAD_CHUNK_PADDING_MS)
        speech_audio = audio[speech_start_ms:speech_end_ms]

        if len(speech_audio) / 1000.0 > MAX_FINAL_DURATION_SEC:
            tqdm.write(f"[Warning] ファイル '{wav_filename}' の検出された発話長 ({len(speech_audio)/1000.0:.2f}秒) が制限を超えています。スキップします。")
            skipped_too_long_count += 1
            continue

        # 環境音の抽出
        noise_before = audio[:speech_start_ms]
        noise_after = audio[speech_end_ms:]
        ambient_noise = noise_before + noise_after

        # 環境音によるパディング処理
        remaining_capacity_sec = MAX_FINAL_DURATION_SEC - (len(speech_audio) / 1000.0)
        
        front_padding_duration_ms = int(random.uniform(0, remaining_capacity_sec / 2) * 1000)
        front_padding = create_random_noise_padding(ambient_noise, front_padding_duration_ms)
        
        current_len_ms = len(front_padding) + len(speech_audio)
        remaining_for_back_ms = (MAX_FINAL_DURATION_SEC * 1000) - current_len_ms
        back_padding_duration_ms = int(random.uniform(0, remaining_for_back_ms))
        back_padding = create_random_noise_padding(ambient_noise, back_padding_duration_ms)

        processed_audio = front_padding + speech_audio + back_padding
        
        speech_start_sec_final = len(front_padding) / 1000.0
        speech_end_sec_final = speech_start_sec_final + (len(speech_audio) / 1000.0)
        
        processed_wav_filepath = os.path.join(processed_wav_output_dir, wav_filename)
        try:
            processed_audio.export(processed_wav_filepath, format="wav")
        except Exception as e:
            tqdm.write(f"[Error] 加工済み音声 '{processed_wav_filepath}' のエクスポート中にエラーが発生しました: {e}。スキップします。")
            skipped_other_error_count += 1
            continue

        try:
            abs_wav_path = os.path.abspath(processed_wav_filepath)
            path_parts = abs_wav_path.split(os.sep)
            dataset_index = path_parts.index('dataset')
            base_dir = os.sep.join(path_parts[:dataset_index])
            path_for_json = os.path.relpath(abs_wav_path, start=base_dir)
        except ValueError:
            path_for_json = os.path.relpath(processed_wav_filepath, start=output_json_dir)
        
        path_for_json = path_for_json.replace(os.sep, '/')
        
        item = {
            "path": path_for_json,
            "timestamps": {
                "speech": [[round(speech_start_sec_final, 3), round(speech_end_sec_final, 3)]]
            },
            "text": transcript
        }
        output_data.append(item)

    # JSONファイルの書き出し
    try:
        with open(output_json_path, 'w', encoding='utf-8') as out_f:
            json.dump(output_data, out_f, ensure_ascii=False, indent=2)
        print(f"\n[Success] JSONファイルを出力しました: {output_json_path}")
    except Exception as e:
        print(f"\n[Error] JSONファイルの書き込み中にエラーが発生しました: {e}")

    # 結果のサマリー表示
    print("\n--- 処理結果 ---")
    print(f"合計処理済みファイル数: {len(output_data)}")
    print(f"  - スキップ (長すぎ): {skipped_too_long_count}")
    print(f"  - スキップ (テキストなし): {skipped_no_transcript_count}")
    print(f"  - スキップ (発話検出不可): {skipped_no_speech_detected}")
    print(f"  - スキップ (その他エラー): {skipped_other_error_count}")
    print("--- 処理完了 ---")


# --- 実行例 (NHK40データセット用) ---
# ご自身の環境に合わせて、以下のパスを修正してください。
speech_wav_input_dir = "./../dataset/NHK40/combined"
transcript_file_path = "./downloaded_folder/ATR_NHK_corpus/NHK40/NHK40.txt"
output_json_path = "./../json/NHK40_speech_labels.json"
processed_wav_output_dir = "./../dataset/NHK40/processed_wavs"

# 関数を呼び出します
print("--- NHK40データセットのJSON生成と音声加工を開始 ---")
generate_speech_json_with_blanks(speech_wav_input_dir, transcript_file_path, output_json_path, processed_wav_output_dir)


/home/tsukagoshitoshihiro/.pyenv/versions/apsipa/lib/python3.10/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


--- NHK40データセットのJSON生成と音声加工を開始 ---
[Info] 書き起こしファイルから 40 行のテキストを読み込みました: ./downloaded_folder/ATR_NHK_corpus/NHK40/NHK40.txt
[Info] 1600 個のWAVファイルを処理します。


JSONデータ生成＆音声加工中: 100%|██████████| 1600/1600 [00:47<00:00, 33.80it/s]


[Success] JSONファイルを出力しました: ./../json/NHK40_speech_labels.json

--- 処理結果 ---
合計処理済みファイル数: 1600
  - スキップ (長すぎ): 0
  - スキップ (テキストなし): 0
  - スキップ (発話検出不可): 0
  - スキップ (その他エラー): 0
--- 処理完了 ---


## HPフィルタ適用

In [15]:
# ==============================================================================
# セル 1: 必要なライブラリのインポート
# ==============================================================================
import os
import numpy as np
from scipy.io import wavfile
from scipy.signal import butter, filtfilt
import tempfile
import logging
import shutil # 異なるデバイス間でのファイル移動に対応

# ==============================================================================
# セル 2: 関数の定義
# ==============================================================================

# ログ設定: 処理の進捗やエラーを分かりやすく表示します
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def apply_highpass_filter_and_overwrite(filepath, cutoff_freq, order=4):
    """
    WAVファイルにハイパスフィルターを適用し、同じファイルに上書き保存する関数
    [Errno 18] Invalid cross-device link エラーに対応済み
    """
    # 一時ファイルは、処理対象ファイルと同じディレクトリに作成することで
    # "Invalid cross-device link" エラーを回避します。
    temp_dir = os.path.dirname(filepath)
    temp_fd, temp_path = tempfile.mkstemp(suffix='.wav', dir=temp_dir)
    os.close(temp_fd)

    try:
        # WAVファイルを読み込む
        sample_rate, data = wavfile.read(filepath)
        original_dtype = data.dtype

        # フィルター係数を計算 (バターワースフィルター)
        nyquist_freq = 0.5 * sample_rate
        normal_cutoff = cutoff_freq / nyquist_freq
        b, a = butter(order, normal_cutoff, btype='high', analog=False)

        # ゼロ位相フィルターを適用 (ステレオ・モノラル両対応)
        if data.ndim == 1:
            filtered_data = filtfilt(b, a, data)
        else:
            filtered_data = np.zeros_like(data, dtype=np.float64)
            for i in range(data.shape[1]):
                filtered_data[:, i] = filtfilt(b, a, data[:, i])

        # 元のデータ型に変換してクリッピングを防ぐ
        if np.issubdtype(original_dtype, np.integer):
            max_val = np.iinfo(original_dtype).max
            min_val = np.iinfo(original_dtype).min
            filtered_data = np.clip(filtered_data, min_val, max_val)

        # 一時ファイルに書き出す
        wavfile.write(temp_path, sample_rate, filtered_data.astype(original_dtype))

        # shutil.move() を使用して、異なるデバイス間でもファイルを安全に移動（上書き）
        shutil.move(temp_path, filepath)

        logging.info(f"上書き完了: {os.path.basename(filepath)}")

    except Exception as e:
        logging.error(f"{os.path.basename(filepath)} の処理に失敗しました。 - {e}")
    finally:
        # 処理が終わったら、念のため一時ファイルが残っていれば削除
        if os.path.exists(temp_path):
            os.remove(temp_path)

# ==============================================================================
# セル 3: 設定と処理の実行
# ==============================================================================

# --- 設定項目 ---
# ★★★ 処理したいWAVファイルがあるフォルダのパスを指定 ★★★
target_folder = "./../dataset/ATR503/processed_wavs"
cutoff_frequency = 100.0  # カットオフ周波数 (Hz)
# ----------------

if not os.path.isdir(target_folder):
    logging.error(f"指定されたフォルダが見つかりません: {target_folder}")
    logging.error("パスが正しいか、Jupyter Notebookファイルからの相対位置を確認してください。")
else:

    logging.info(f"対象フォルダ: {target_folder}")
    
    logging.info("処理を開始します...")
    
    file_count = 0
    # フォルダ内のファイルをループ処理
    for filename in tqdm(os.listdir(target_folder)):
        # .wavファイルのみを対象とする
        if filename.lower().endswith(".wav"):
            filepath = os.path.join(target_folder, filename)
            # 上で定義したフィルター適用・上書き関数を呼び出す
            apply_highpass_filter_and_overwrite(filepath, cutoff_frequency)
            file_count += 1
            
    if file_count == 0:
        logging.warning("処理対象の.wavファイルが見つかりませんでした。")
    
    logging.info("\n----------------------------------------------------")
    logging.info("すべての処理が完了しました。")
    logging.info(f"合計 {file_count} 個のファイルを処理しました。")
    logging.info("----------------------------------------------------")


    

2025-07-15 05:45:53,274 - INFO - 対象フォルダ: ./../dataset/ATR503/processed_wavs
2025-07-15 05:45:53,274 - INFO - 処理を開始します...
  0%|          | 0/20120 [00:00<?, ?it/s]2025-07-15 05:45:53,283 - INFO - 上書き完了: ATR503K_div_MKA01_speech07_008.wav
2025-07-15 05:45:53,285 - INFO - 上書き完了: ATR503L_div_MHT02_speech02_024.wav
2025-07-15 05:45:53,289 - INFO - 上書き完了: ATR503L_div_MYN02_speech02_038.wav
2025-07-15 05:45:53,291 - INFO - 上書き完了: ATR503K_div_MTH01_speech09_045.wav
2025-07-15 05:45:53,293 - INFO - 上書き完了: ATR503K_div_MTH01_speech05_011.wav
2025-07-15 05:45:53,296 - INFO - 上書き完了: ATR503K_div_MYK02_speech06_022.wav
2025-07-15 05:45:53,299 - INFO - 上書き完了: ATR503K_div_MTH01_speech01_017.wav
2025-07-15 05:45:53,302 - INFO - 上書き完了: ATR503K_div_MTK01_speech01_041.wav
2025-07-15 05:45:53,304 - INFO - 上書き完了: ATR503K_div_MTH01_speech05_001.wav
2025-07-15 05:45:53,305 - INFO - 上書き完了: ATR503K_div_MKH01_speech04_027.wav
2025-07-15 05:45:53,308 - INFO - 上書き完了: ATR503K_div_FMT01_speech05_004.wav
2025-07-15 05

In [16]:

# --- 設定項目 ---
# ★★★ 処理したいWAVファイルがあるフォルダのパスを指定 ★★★
target_folder = "./../dataset/NHK40/processed_wavs"
cutoff_frequency = 100.0  # カットオフ周波数 (Hz)
# ----------------

if not os.path.isdir(target_folder):
    logging.error(f"指定されたフォルダが見つかりません: {target_folder}")
    logging.error("パスが正しいか、Jupyter Notebookファイルからの相対位置を確認してください。")
else:

    logging.info(f"対象フォルダ: {target_folder}")
    
    logging.info("処理を開始します...")
    
    file_count = 0
    # フォルダ内のファイルをループ処理
    for filename in tqdm(os.listdir(target_folder)):
        # .wavファイルのみを対象とする
        if filename.lower().endswith(".wav"):
            filepath = os.path.join(target_folder, filename)
            # 上で定義したフィルター適用・上書き関数を呼び出す
            apply_highpass_filter_and_overwrite(filepath, cutoff_frequency)
            file_count += 1
            
    if file_count == 0:
        logging.warning("処理対象の.wavファイルが見つかりませんでした。")
    
    logging.info("\n----------------------------------------------------")
    logging.info("すべての処理が完了しました。")
    logging.info(f"合計 {file_count} 個のファイルを処理しました。")
    logging.info("----------------------------------------------------")

2025-07-15 05:46:41,107 - INFO - 対象フォルダ: ./../dataset/NHK40/processed_wavs
2025-07-15 05:46:41,108 - INFO - 処理を開始します...
  0%|          | 0/1600 [00:00<?, ?it/s]2025-07-15 05:46:41,112 - INFO - 上書き完了: NHK40L_div_MKK03_speech00_005.wav
2025-07-15 05:46:41,115 - INFO - 上書き完了: NHK40L_div_MYK03_speech00_003.wav
2025-07-15 05:46:41,117 - INFO - 上書き完了: NHK40L_div_MYN02_speech00_020.wav
2025-07-15 05:46:41,120 - INFO - 上書き完了: NHK40K_div_MTN02_speech00_021.wav
2025-07-15 05:46:41,123 - INFO - 上書き完了: NHK40K_div_MRY01_speech00_017.wav
2025-07-15 05:46:41,125 - INFO - 上書き完了: NHK40K_div_MMT01_speech00_039.wav
2025-07-15 05:46:41,127 - INFO - 上書き完了: NHK40K_div_MYM01_speech00_014.wav
2025-07-15 05:46:41,129 - INFO - 上書き完了: NHK40L_div_FMH01_speech00_020.wav
2025-07-15 05:46:41,131 - INFO - 上書き完了: NHK40K_div_FMT01_speech00_003.wav
2025-07-15 05:46:41,134 - INFO - 上書き完了: NHK40K_div_MAS01_speech00_016.wav
2025-07-15 05:46:41,136 - INFO - 上書き完了: NHK40K_div_MYS04_speech00_029.wav
2025-07-15 05:46:41,139 - 